# <font color="#418FDE" size="6.5" uppercase>**LeNet mit Keras**</font>

>Last update: 20260825.
    
By the end of this Lecture, you will be able to:
- Bereiten kleine Bildtensoren für TensorFlow-CNNs reproduzierbar vor. 
- Erstellen und trainieren ein LeNet-ähnliches CNN mit wenigen Epochen. 
- Bewerten CNN-Ergebnisse mit Lernkurven, Konfusionsmatrix und Fehlklassifikationen. 


## **1. Bildtensoren vorbereiten**

### **1.1. Daten laden**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_16/Lecture_A/image_01_01.jpg?v=1787658166" width="250">



>* Einheitliche Bildtensoren aus Rohbildern erstellen
>* Form, Kanäle und Datentypen passend vorbereiten

>* Datenaufteilung kontrolliert und reproduzierbar festlegen
>* Ergebnisse durch konsistente Stichproben vergleichbar machen

>* Bildanzahl, Größen und Kanäle früh prüfen
>* Bilder und Labels korrekt zuordnen



In [ ]:
#@title Python-Code - Daten laden

# Wir laden kleine Ziffernbilder reproduzierbar als Tensoren.
# Formen, Kanäle und Labels werden sichtbar geprüft.
# Am Ende sehen wir Beispielbilder mit korrekten Dimensionen.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

# Der Datensatz enthält kleine Graustufenbilder von Ziffern.
digits = load_digits()
images_uint8 = digits.images.astype(np.uint8)
labels = digits.target.astype(np.int64)

# Für TensorFlow-CNNs ergänzen wir die Kanalachse.
image_tensors = images_uint8[..., np.newaxis]

# Diese Prüfungen schützen vor vertauschten oder unpassenden Daten.
if image_tensors.shape[0] != labels.shape[0]:
    raise ValueError("Bilder und Labels haben unterschiedliche Anzahlen.")

if image_tensors.ndim != 4:
    raise ValueError("Der Bildtensor muss vier Dimensionen besitzen.")

# Die Aufteilung ist durch random_state reproduzierbar.
train_images, test_images, train_labels, test_labels = train_test_split(
    image_tensors,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels,
)

# Erst nach dem Laden skalieren wir Pixelwerte für das Modell.
train_images_float = train_images.astype(np.float32) / 16.0
test_images_float = test_images.astype(np.float32) / 16.0

print(f"Geladene Bilder: {image_tensors.shape[0]}")
print(f"Tensorform gesamt: {image_tensors.shape}")
print(f"Trainingsform: {train_images_float.shape}")
print(f"Testform: {test_images_float.shape}")
print(f"Pixelbereich nach Skalierung: {train_images_float.min():.1f} bis {train_images_float.max():.1f}")

# Ein sichtbares Beispiel verbindet Tensorform und Label.
example_index = 0
example_image = train_images_float[example_index, :, :, 0]
example_label = train_labels[example_index]

fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(example_image, cmap="gray", vmin=0.0, vmax=1.0)
ax.set_title(f"Geladenes Beispiel, Label: {example_label}")
ax.set_xlabel("Pixelspalte")
ax.set_ylabel("Pixelzeile")
plt.show()



### **1.2. Pixelwerte normalisieren**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_16/Lecture_A/image_01_02.jpg?v=1787658162" width="250">



>* Pixelwerte auf 0 bis 1 skalieren
>* Stabileres Lernen aus kleinen Graustufenbildern

>* Bildinhalt bleibt erhalten, Werte werden günstiger
>* CNNs lernen stabiler und schneller interpretierbar

>* Gleiche Skalierung für alle Datensplits
>* Vergleiche bleiben dadurch zuverlässig interpretierbar



In [ ]:
#@title Python-Code - Pixelwerte normalisieren

# Wir normalisieren kleine Bildtensoren für CNNs.
# Pixelwerte werden von 0 bis 1 skaliert.
# Die Ausgabe zeigt gleiche Struktur, kleinere Zahlen.

import numpy as np
import matplotlib.pyplot as plt

# Dieses synthetische Graustufenbild bleibt klein und übersichtlich.
image_uint8 = np.array(
    [[0, 32, 64, 96], [128, 160, 192, 224], [255, 200, 150, 100]],
    dtype=np.uint8,
)

# Die Form wird geprüft, bevor wir weiterarbeiten.
if image_uint8.ndim != 2:
    raise ValueError("Erwartet wird ein einzelnes Graustufenbild.")

# Für TensorFlow-CNNs ergänzen wir eine Kanalachse.
image_tensor = image_uint8[..., np.newaxis]

# Die Normalisierung erzeugt Fließkommazahlen zwischen 0 und 1.
image_normalized = image_tensor.astype("float32") / 255.0

# Diese Werte zeigen den Effekt ohne große Arrays auszugeben.
print(f"Ursprünglicher Datentyp: {image_tensor.dtype}")
print(f"Normalisierter Datentyp: {image_normalized.dtype}")
print(f"Tensorform für CNNs: {image_normalized.shape}")
print(f"Wertebereich vorher: {image_tensor.min()} bis {image_tensor.max()}")
print(f"Wertebereich nachher: {image_normalized.min():.2f} bis {image_normalized.max():.2f}")

# Die Bildstruktur bleibt trotz anderer Zahlenskala erhalten.
fig, ax = plt.subplots(figsize=(5, 3))
ax.imshow(image_normalized[:, :, 0], cmap="gray", vmin=0, vmax=1)
ax.set_title("Normalisierte Pixelwerte eines synthetischen Bildes")
ax.set_xlabel("Spalte")
ax.set_ylabel("Zeile")
plt.show()



### **1.3. Feature Maps verstehen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_16/Lecture_A/image_01_03.jpg?v=1787658164" width="250">



>* Feature Maps zeigen erkannte Musterpositionen
>* Sie interpretieren Bildtensoren durch Faltungsfilter

>* Tensorform, Skalierung und Kanäle korrekt vorbereiten
>* Saubere Daten erzeugen stabile Feature Maps

>* Tiefere Schichten erkennen abstraktere Bildmuster
>* Feature Maps zeigen schrittweise Merkmalsgewinnung



In [ ]:
#@title Python-Code - Feature Maps verstehen

# Wir erzeugen einen kleinen Bildtensor.
# Ein Filter findet einfache Kanten.
# Die Feature Map zeigt starke Reaktionen.

import numpy as np
import matplotlib.pyplot as plt

# Dieses synthetische Bild bleibt klein und übersichtlich.
image = np.zeros((8, 8), dtype=np.float32)
image[:, 3:] = 1.0

# TensorFlow-CNNs erwarten oft Höhe, Breite und Kanal.
image_tensor = image.reshape(8, 8, 1)

# Dieser Filter reagiert auf vertikale Helligkeitswechsel.
vertical_edge_filter = np.array(
    [[-1.0, 0.0, 1.0], [-1.0, 0.0, 1.0], [-1.0, 0.0, 1.0]],
    dtype=np.float32,
)

# Die Feature Map entsteht durch Faltung ohne Padding.
feature_map = np.zeros((6, 6), dtype=np.float32)
for row in range(6):
    for col in range(6):
        patch = image[row:row + 3, col:col + 3]
        feature_map[row, col] = np.sum(patch * vertical_edge_filter)

# Eine einfache Prüfung macht die Tensorform sichtbar.
if image_tensor.shape != (8, 8, 1):
    raise ValueError("Der Bildtensor hat nicht die erwartete Form.")

print(f"Bildtensor-Form: {image_tensor.shape}")
print(f"Filter-Form: {vertical_edge_filter.shape}")
print(f"Feature-Map-Form: {feature_map.shape}")
print(f"Stärkste Reaktion: {feature_map.max():.1f}")

# Die Grafik zeigt die Feature Map als Wärmekarte.
fig, ax = plt.subplots(figsize=(5, 4))
heatmap = ax.imshow(feature_map, cmap="viridis", vmin=0.0, vmax=3.0)
fig.colorbar(heatmap, ax=ax, label="Filterreaktion")

ax.set_title("Feature Map eines vertikalen Kantenfilters")
ax.set_xlabel("Spaltenposition")
ax.set_ylabel("Zeilenposition")
plt.show()



## **2. CNN Aufbau**

### **2.1. Padding und Stride**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_16/Lecture_A/image_02_01.jpg?v=1787658156" width="250">



>* Padding erhält Randinformationen und Bildgröße.
>* So bleiben lokale Muster länger nutzbar.

>* Kleiner Stride liefert detaillierte Merkmalskarten.
>* Pooling übernimmt meist die Verkleinerung.

>* Same erhält Auflösung, valid verdichtet stärker
>* Gute Wahl schützt Details und Lernzeit



In [ ]:
#@title Python-Code - Padding und Stride

# Wir untersuchen Padding und Stride anschaulich.
# Ein kleiner Bildtensor macht Größenänderungen sichtbar.
# Die Ausgabe zeigt resultierende Merkmalskarten.

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

# Ein synthetisches Bild enthält ein helles Kreuz.
image = np.zeros((1, 7, 7, 1), dtype=np.float32)
image[0, 3, :, 0] = 1.0
image[0, :, 3, 0] = 1.0

# Ein einfacher Filter zählt lokale Helligkeit.
kernel = np.ones((3, 3, 1, 1), dtype=np.float32)

# Drei Faltungen zeigen unterschiedliche Architekturentscheidungen.
same_stride_one = tf.nn.conv2d(image, kernel, strides=1, padding="SAME")
valid_stride_one = tf.nn.conv2d(image, kernel, strides=1, padding="VALID")
valid_stride_two = tf.nn.conv2d(image, kernel, strides=2, padding="VALID")

# Die Formen zeigen, wie stark die Merkmalskarten schrumpfen.
print("Eingabebild: 7x7 Pixel")
print(f"SAME, Stride 1: {same_stride_one.shape[1]}x{same_stride_one.shape[2]}")
print(f"VALID, Stride 1: {valid_stride_one.shape[1]}x{valid_stride_one.shape[2]}")
print(f"VALID, Stride 2: {valid_stride_two.shape[1]}x{valid_stride_two.shape[2]}")

# Für die Grafik werden die drei Ergebnisse nebeneinandergelegt.
canvas = np.full((7, 23), np.nan, dtype=np.float32)
canvas[:, 0:7] = same_stride_one.numpy()[0, :, :, 0]
canvas[1:6, 9:14] = valid_stride_one.numpy()[0, :, :, 0]
canvas[2:5, 17:20] = valid_stride_two.numpy()[0, :, :, 0]

# Eine einzige Achse vergleicht die räumlichen Ausgaben.
fig, ax = plt.subplots(figsize=(8, 3))
image_plot = ax.imshow(canvas, cmap="viridis", vmin=0, vmax=5)
ax.set_title("Padding und Stride verändern die Größe der Merkmalskarte")
ax.set_xlabel("Nebeneinander: SAME s=1, VALID s=1, VALID s=2")
ax.set_ylabel("Pixelposition")
ax.set_xticks([])
ax.set_yticks([])
fig.colorbar(image_plot, ax=ax, label="Filterantwort")
plt.show()



### **2.2. MaxPooling verstehen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_16/Lecture_A/image_02_02.jpg?v=1787658158" width="250">



>* MaxPooling behält stärkste lokale Merkmale
>* Bilder werden kompakter und robuster dargestellt

>* Kleinere Merkmalskarten sparen Rechenaufwand
>* Robuster gegen kleine Bildverschiebungen

>* Pooling entfernt Details, manchmal auch wichtige.
>* Gezielt eingesetzt macht es LeNet robuster.



In [ ]:
#@title Python-Code - MaxPooling verstehen

# Dieses Beispiel zeigt MaxPooling mit kleinen Zahlen.
# Lokale Maxima verdichten eine Merkmalskarte sichtbar.
# Die Ausgabe vergleicht Eingabe und Pooling-Ergebnis.

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

# Diese synthetische Merkmalskarte bleibt bewusst sehr klein.
feature_map = np.array(
    [[1, 2, 0, 1], [3, 8, 2, 0], [0, 1, 5, 4], [2, 1, 3, 9]],
    dtype=np.float32,
)

# TensorFlow erwartet Bilddaten als Stapel mit Kanalachse.
input_tensor = feature_map.reshape(1, 4, 4, 1)
if input_tensor.shape != (1, 4, 4, 1):
    raise ValueError("Die Tensorform passt nicht zum Pooling-Beispiel.")

# MaxPooling nimmt aus jedem Zwei-mal-zwei-Bereich den größten Wert.
pooling_layer = tf.keras.layers.MaxPooling2D(pool_size=(2, 2), strides=(2, 2))
pooled_tensor = pooling_layer(input_tensor)
pooled_map = pooled_tensor.numpy().reshape(2, 2)

# Die kurzen Ausgaben zeigen Formänderung und erhaltene Maxima.
print("Eingabeform: 4 x 4 x 1")
print("Ausgabeform nach MaxPooling: 2 x 2 x 1")
print("Pooling-Ergebnis: [[8, 2], [2, 9]]")

# Die Grafik zeigt die verdichtete Merkmalskarte als Heatmap.
fig, ax = plt.subplots(figsize=(4, 4))
image = ax.imshow(pooled_map, cmap="viridis", vmin=0, vmax=9)

# Zahlen in den Zellen machen die Maxima direkt ablesbar.
for row in range(pooled_map.shape[0]):
    for col in range(pooled_map.shape[1]):
        ax.text(col, row, int(pooled_map[row, col]), ha="center", va="center")

# Achsenbeschriftungen betonen die kleinere räumliche Auflösung.
ax.set_title("MaxPooling: stärkste Aktivierungen bleiben")
ax.set_xlabel("Spalte nach Pooling")
ax.set_ylabel("Zeile nach Pooling")

# Die Farbleiste erklärt die Aktivierungsstärke.
fig.colorbar(image, ax=ax, label="Aktivierung")
plt.show()



### **2.3. LeNet Architektur**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_16/Lecture_A/image_02_03.jpg?v=1787658160" width="250">



>* LeNet verarbeitet Bilder schrittweise mit CNN-Schichten
>* Kompakt, lehrreich und ideal für kleine Bildtensoren

>* Filter lernen einfache visuelle Muster.
>* Pooling hält Training kompakt und effizient.

>* Dichte Schichten bündeln Merkmale zur Klassenvorhersage
>* Schlanke Keras-Modelle passen zu kleinen Bildern



In [ ]:
#@title Python-Code - LeNet Architektur

# Wir bauen ein kleines LeNet-Modell.
# Die Schichten zeigen den typischen CNN-Fluss.
# Die Ausgabe fasst Formen und Parameter zusammen.

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

# Feste Startwerte machen das Beispiel reproduzierbar.
np.random.seed(42)
tf.random.set_seed(42)

# Kleine synthetische Graustufenbilder ersetzen einen Datendownload.
image_count = 120
image_size = 28

# Jede Klasse erhält ein einfaches, sichtbares Muster.
images = np.zeros((image_count, image_size, image_size, 1), dtype=np.float32)
labels = np.zeros(image_count, dtype=np.int32)

for index in range(image_count):
    label = index % 3
    labels[index] = label
    images[index, :, :, 0] = np.random.default_rng(index).normal(
        0.05, 0.02, (image_size, image_size)
    )

    if label == 0:
        images[index, 6:22, 12:16, 0] += 0.9
    elif label == 1:
        images[index, 12:16, 6:22, 0] += 0.9
    else:
        images[index, 7:21, 7:21, 0] += np.eye(14) * 0.9

images = np.clip(images, 0.0, 1.0)

# Die Form entspricht kleinen TensorFlow-Bildtensoren.
if images.shape != (120, 28, 28, 1):
    raise ValueError("Die Bildtensoren haben nicht die erwartete Form.")

# Dieses Modell folgt der kompakten LeNet-Idee.
model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(28, 28, 1)),
        tf.keras.layers.Conv2D(6, kernel_size=5, activation="relu"),
        tf.keras.layers.MaxPooling2D(pool_size=2),
        tf.keras.layers.Conv2D(16, kernel_size=5, activation="relu"),
        tf.keras.layers.MaxPooling2D(pool_size=2),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.Dense(3, activation="softmax"),
    ]
)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

# Wenige Epochen reichen für dieses einfache Musterbeispiel.
history = model.fit(images, labels, epochs=3, batch_size=16, verbose=0)

# Wir betrachten die wichtigsten Architekturstationen.
layer_names = [layer.name for layer in model.layers]
output_shapes = [layer.output.shape for layer in model.layers]

print(f"Tensorform der Eingabe: {images.shape}")
print(f"Anzahl Klassen: {len(np.unique(labels))}")
print(f"Trainierbare Parameter: {model.count_params()}")
print(f"Letzte Trainingsgenauigkeit: {history.history['accuracy'][-1]:.2f}")
print(f"Erste Schicht: {layer_names[0]} -> {output_shapes[0]}")
print(f"Letzte Schicht: {layer_names[-1]} -> {output_shapes[-1]}")

# Die Lernkurve zeigt, ob das kleine CNN lernt.
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(history.history["accuracy"], marker="o", label="Training")
ax.set_title("LeNet-ähnliches CNN: Lernkurve")
ax.set_xlabel("Epoche")
ax.set_ylabel("Genauigkeit")
ax.set_ylim(0, 1.05)
ax.legend()
plt.show()



## **3. Training und Auswertung**

### **3.1. Training auf CPU**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_16/Lecture_A/image_03_01.jpg?v=1787658168" width="250">



>* CPU-Training macht CNN-Lernen gut nachvollziehbar
>* Langsameres Training fördert bewusste Modellentscheidungen

>* Lernkurven zeigen frühe Trainingsprobleme
>* CPU-Auswertung diagnostiziert Generalisierung und Overfitting

>* Konfusionsmatrix zeigt typische Klassenverwechslungen.
>* Fehlklassifikationen erklären Grenzen und Verbesserungen.



In [ ]:
#@title Python-Code - Training auf CPU

# Wir trainieren ein kleines CNN bewusst auf CPU.
# Lernkurven und Fehler zeigen die Modellqualität.
# Die Grafik macht typische Verwechslungen sichtbar.

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.datasets import load_digits
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split

# Feste Startwerte machen das kurze Training reproduzierbarer.
np.random.seed(42)
tf.random.set_seed(42)

# Der Digits-Datensatz enthält kleine Graustufenbilder.
digits = load_digits()
images = digits.images.astype("float32")
labels = digits.target.astype("int64")

# Diese Prüfung schützt vor unerwarteten Datenformen.
if images.shape[1:] != (8, 8):
    raise ValueError("Erwartet werden 8x8-Graustufenbilder.")

# TensorFlow erwartet einen Kanal für Graustufenbilder.
images = images[..., np.newaxis] / 16.0
class_names = [str(number) for number in range(10)]

# Die Aufteilung trennt Training und spätere Bewertung.
train_images, test_images, train_labels, test_labels = train_test_split(
    images, labels, test_size=0.25, random_state=42, stratify=labels
)

# Ein kleiner Validierungsanteil erzeugt Lernkurven während des Trainings.
train_images, val_images, train_labels, val_labels = train_test_split(
    train_images, train_labels, test_size=0.2, random_state=42, stratify=train_labels
)

# Dieses LeNet-ähnliche Modell bleibt für CPU-Training kompakt.
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(8, 8, 1)),
    tf.keras.layers.Conv2D(8, kernel_size=3, activation="relu", padding="same"),
    tf.keras.layers.AveragePooling2D(pool_size=2),
    tf.keras.layers.Conv2D(16, kernel_size=3, activation="relu", padding="same"),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax"),
])

# Sparse Labels passen direkt zu den Ziffernklassen.
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

# Wenige Epochen reichen für eine erste Diagnose.
history = model.fit(
    train_images,
    train_labels,
    validation_data=(val_images, val_labels),
    epochs=5,
    batch_size=32,
    verbose=0,
)

# Vorhersagen auf Testdaten zeigen die Generalisierung.
probabilities = model.predict(test_images, verbose=0)
predicted_labels = np.argmax(probabilities, axis=1)

# Kennzahlen und Fehlerbeispiele fassen die Auswertung zusammen.
test_accuracy = accuracy_score(test_labels, predicted_labels)
confusion = confusion_matrix(test_labels, predicted_labels)
wrong_indices = np.where(predicted_labels != test_labels)[0]

print(f"TensorFlow-Version: {tf.__version__}")
print(f"Testgenauigkeit nach 5 Epochen: {test_accuracy:.3f}")
print(f"Anzahl Fehlklassifikationen: {len(wrong_indices)} von {len(test_labels)}")

# Die häufigste Verwechslung ignoriert die Diagonale.
confusion_without_diagonal = confusion.copy()
np.fill_diagonal(confusion_without_diagonal, 0)

most_confused_index = np.unravel_index(
    np.argmax(confusion_without_diagonal), confusion_without_diagonal.shape
)
most_confused_count = confusion_without_diagonal[most_confused_index]

print(
    f"Häufigste Verwechslung: {most_confused_index[0]} als "
    f"{most_confused_index[1]} ({most_confused_count} mal)"
)

# Die Grafik zeigt Lernkurven und markiert die Testgenauigkeit.
fig, ax = plt.subplots(figsize=(7, 4))
epochs = np.arange(1, len(history.history["accuracy"]) + 1)

ax.plot(epochs, history.history["accuracy"], marker="o", label="Training")
ax.plot(epochs, history.history["val_accuracy"], marker="o", label="Validierung")
ax.axhline(test_accuracy, color="gray", linestyle="--", label="Test")

ax.set_title("CPU-Training: Lernkurven eines kleinen CNN")
ax.set_xlabel("Epoche")
ax.set_ylabel("Genauigkeit")
ax.set_ylim(0.0, 1.05)

ax.legend()
plt.tight_layout()
plt.show()



### **3.2. Regularisierung gegen Overfitting**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_16/Lecture_A/image_03_02.jpg?v=1787658170" width="250">



>* Regularisierung schützt CNNs vor Auswendiglernen.
>* Validierungsleistung zeigt echte Generalisierung.

>* Datenerweiterung erzeugt sinnvolle Bildvarianten
>* Dropout und kleinere Modelle fördern robuste Merkmale

>* Regularisierung mit Lernkurven und Konfusionsmatrix prüfen
>* Fehler analysieren und Maßnahmen iterativ verbessern



In [ ]:
#@title Python-Code - Regularisierung gegen Overfitting

# Wir vergleichen Regularisierung bei kleinen Ziffernbildern.
# Dropout soll Overfitting im CNN verringern.
# Lernkurven zeigen die Generalisierung sichtbar.

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Feste Startwerte machen das Beispiel reproduzierbar.
np.random.seed(42)
tf.random.set_seed(42)

# Der Datensatz enthält kleine Graustufenbilder von Ziffern.
digits = load_digits()
images = digits.images.astype("float32") / 16.0
labels = digits.target.astype("int64")

# Diese Prüfung macht die erwartete Bildform explizit.
if images.shape[1:] != (8, 8):
    raise ValueError("Erwartet werden 8x8-Bilder.")

# TensorFlow erwartet zusätzlich einen Kanal für Graustufenbilder.
images = images[..., np.newaxis]

# Die Aufteilung trennt Training und Validierung sauber.
train_images, val_images, train_labels, val_labels = train_test_split(
    images, labels, test_size=0.25, stratify=labels, random_state=42
)

# Dieses kleine CNN kann ohne Regularisierung leichter auswendig lernen.
def build_model(use_dropout):
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Input(shape=(8, 8, 1)))
    model.add(tf.keras.layers.Conv2D(16, 3, activation="relu", padding="same"))
    model.add(tf.keras.layers.Flatten())
    if use_dropout:
        model.add(tf.keras.layers.Dropout(0.35))
    model.add(tf.keras.layers.Dense(10, activation="softmax"))
    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

# Beide Modelle lernen mit denselben Daten und Epochen.
plain_model = build_model(use_dropout=False)
regularized_model = build_model(use_dropout=True)

# Verbose null verhindert lange Trainingsausgaben.
plain_history = plain_model.fit(
    train_images, train_labels, epochs=12, batch_size=32,
    validation_data=(val_images, val_labels), verbose=0
)

# Dropout ist nur während des Trainings aktiv.
regularized_history = regularized_model.fit(
    train_images, train_labels, epochs=12, batch_size=32,
    validation_data=(val_images, val_labels), verbose=0
)

# Die Validierungsgenauigkeit bewertet unbekannte Bilder.
plain_predictions = np.argmax(plain_model.predict(val_images, verbose=0), axis=1)
regularized_predictions = np.argmax(
    regularized_model.predict(val_images, verbose=0), axis=1
)

# Die Lücke zeigt den Abstand zwischen Training und Validierung.
plain_gap = plain_history.history["accuracy"][-1] - plain_history.history["val_accuracy"][-1]
regularized_gap = regularized_history.history["accuracy"][-1] - regularized_history.history["val_accuracy"][-1]

print(f"TensorFlow-Version: {tf.__version__}")
print(f"Ohne Dropout: Validierungsgenauigkeit {accuracy_score(val_labels, plain_predictions):.3f}, Lücke {plain_gap:.3f}")
print(f"Mit Dropout: Validierungsgenauigkeit {accuracy_score(val_labels, regularized_predictions):.3f}, Lücke {regularized_gap:.3f}")

# Eine einzelne Achse vergleicht die wichtigsten Lernkurven.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(plain_history.history["val_accuracy"], label="ohne Dropout")
ax.plot(regularized_history.history["val_accuracy"], label="mit Dropout")
ax.set_title("Regularisierung: Validierungsgenauigkeit pro Epoche")
ax.set_xlabel("Epoche")
ax.set_ylabel("Validierungsgenauigkeit")
ax.legend()
plt.show()



### **3.3. CNN Mini Projekt**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_16/Lecture_A/image_03_03.jpg?v=1787658172" width="250">



>* Lernkurven zeigen Trainings- und Validierungsverhalten
>* Abweichende Kurven warnen vor Overfitting

>* Konfusionsmatrix zeigt gute und verwechselte Klassen
>* Fehlermuster helfen, Ursachen gezielt zu prüfen

>* Fehlklassifikationen zeigen konkrete Modellgrenzen.
>* Auswertung führt zu begründeten Verbesserungen.



In [ ]:
#@title Python-Code - CNN Mini Projekt

# Dieses Mini Projekt bewertet ein kleines CNN.
# Lernkurven und Fehler zeigen Modellgrenzen.
# Die Grafik markiert typische Fehlklassifikationen.

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix

# Feste Startwerte machen das Training reproduzierbarer.
np.random.seed(42)
tf.random.set_seed(42)

# Der Digits Datensatz enthält kleine Graustufenbilder.
digits = load_digits()
images = digits.images.astype("float32")
labels = digits.target.astype("int64")

# Diese Prüfung schützt vor unerwarteten Datenformen.
if images.shape[1:] != (8, 8):
    raise ValueError("Erwartet werden Bilder mit 8 mal 8 Pixeln.")

# TensorFlow erwartet einen zusätzlichen Kanal für Graustufenbilder.
images = images / 16.0
images = images[..., np.newaxis]

# Die Aufteilung bleibt klassenbalanciert und reproduzierbar.
train_images, test_images, train_labels, test_labels = train_test_split(
    images, labels, test_size=0.25, random_state=42, stratify=labels
)

# Ein kleines LeNet ähnliches CNN reicht hier aus.
model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(8, 8, 1)),
        tf.keras.layers.Conv2D(8, 3, activation="relu", padding="same"),
        tf.keras.layers.AveragePooling2D(pool_size=2),
        tf.keras.layers.Conv2D(16, 3, activation="relu", padding="same"),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.Dense(10, activation="softmax"),
    ]
)

# Die Kompilierung legt Lernziel und Kennzahl fest.
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

# Wenige Epochen genügen für eine schnelle Auswertung.
history = model.fit(
    train_images,
    train_labels,
    validation_split=0.2,
    epochs=5,
    batch_size=32,
    verbose=0,
)

# Vorhersagen werden in Klassenlabels umgewandelt.
probabilities = model.predict(test_images, verbose=0)
predicted_labels = np.argmax(probabilities, axis=1)

# Drei Kennzahlen fassen die Auswertung knapp zusammen.
test_accuracy = accuracy_score(test_labels, predicted_labels)
confusion = confusion_matrix(test_labels, predicted_labels)
wrong_indices = np.where(predicted_labels != test_labels)[0]

# Die häufigste Verwechslung wird aus der Matrix gelesen.
confusion_without_diagonal = confusion.copy()
np.fill_diagonal(confusion_without_diagonal, 0)
most_confused = np.unravel_index(
    np.argmax(confusion_without_diagonal), confusion_without_diagonal.shape
)

print(f"scikit-learn Version: 1.9.0")
print(f"Testgenauigkeit: {test_accuracy:.3f}")
print(f"Fehlklassifikationen im Testset: {len(wrong_indices)} von {len(test_labels)}")
print(f"Häufigste Verwechslung: {most_confused[0]} als {most_confused[1]}")

# Die Grafik verbindet Lernkurve und konkrete Fehlerdiagnose.
fig, ax = plt.subplots(figsize=(7, 4))
epochs = np.arange(1, len(history.history["accuracy"]) + 1)

ax.plot(epochs, history.history["accuracy"], marker="o", label="Training")
ax.plot(epochs, history.history["val_accuracy"], marker="o", label="Validierung")

ax.set_title("CNN Mini Projekt: Lernkurven prüfen")
ax.set_xlabel("Epoche")
ax.set_ylabel("Genauigkeit")
ax.set_ylim(0.0, 1.05)
ax.legend()

plt.show()



# <font color="#418FDE" size="6.5" uppercase>**LeNet mit Keras**</font>


In this lecture, you learned to:
- Bereiten kleine Bildtensoren für TensorFlow-CNNs reproduzierbar vor. 
- Erstellen und trainieren ein LeNet-ähnliches CNN mit wenigen Epochen. 
- Bewerten CNN-Ergebnisse mit Lernkurven, Konfusionsmatrix und Fehlklassifikationen. 

In the next Lecture (Lecture B), we will go over 'Transfer mit Keras'